# Notebook 1 (v4) — SG Data Generation with Qwen2.5-3B-Instruct

**Why model changed from Phi-4-mini:**
- Phi-4-mini ignored `Step N:` format completely → 0% valid rate
- It is a reasoning model, not an instruction-following model
- Qwen2.5-3B-Instruct is specifically trained for strict format compliance

**Expected valid rate: 70-85%**

**Memory: ~6GB float16 on P100 — safe without 4-bit**

In [1]:
# ── CELL 1: Install ───────────────────────────────────────────
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# ── CELL 2: HuggingFace Login ─────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
print("HuggingFace login successful.")

HuggingFace login successful.


In [3]:
# ── CELL 3: Imports + GPU Check ───────────────────────────────
import os, json, re, random, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/sg_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 17.1 GB


In [4]:
# ── CELL 4: Config ────────────────────────────────────────────
# num_samples: 50 = test | 3000 = full run

CONFIG = {
    "model_name"         : "Qwen/Qwen2.5-3B-Instruct",  # switched from Phi-4-mini
    "num_samples"        : 3000,      # ← change to 3000 for full run
    "random_seed"        : 42,
    "max_new_tokens"     : 150,     # short steps = fewer tokens needed
    "temperature"        : 0.1,     # low for strict format
    "do_sample"          : True,
    "batch_size"         : 8,       # Qwen2.5-3B is smaller → bigger batch ok
    "max_words_per_step" : 20,      # reject steps longer than this
    "raw_output_file"    : f"{OUTPUT_DIR}/sg_raw.jsonl",
    "valid_output_file"  : f"{OUTPUT_DIR}/sg_valid.jsonl",
    "checkpoint_file"    : f"{OUTPUT_DIR}/checkpoint.json",
    "report_file"        : f"{OUTPUT_DIR}/generation_report.json",
    "save_every"         : 50,
}

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")

Config:
  model_name            : Qwen/Qwen2.5-3B-Instruct
  num_samples           : 3000
  random_seed           : 42
  max_new_tokens        : 150
  temperature           : 0.1
  do_sample             : True
  batch_size            : 8
  max_words_per_step    : 20
  raw_output_file       : /kaggle/working/sg_data/sg_raw.jsonl
  valid_output_file     : /kaggle/working/sg_data/sg_valid.jsonl
  checkpoint_file       : /kaggle/working/sg_data/checkpoint.json
  report_file           : /kaggle/working/sg_data/generation_report.json
  save_every            : 50


In [5]:
# ── CELL 5: Load GSM8K ───────────────────────────────────────
print("Loading GSM8K...")
gsm8k      = load_dataset("gsm8k", "main")
train_data = gsm8k["train"]

random.seed(CONFIG["random_seed"])
indices      = random.sample(range(len(train_data)), CONFIG["num_samples"])
sampled_data = [train_data[i] for i in indices]

print(f"GSM8K size : {len(train_data)}")
print(f"Sampled    : {len(sampled_data)}")
print(f"\nExample:")
print(sampled_data[0]["question"])

Loading GSM8K...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K size : 7473
Sampled    : 3000

Example:
For every 12 cans you recycle, you receive $0.50, and for every 5 kilograms of newspapers, you receive $1.50. If your family collected 144 cans and 20 kilograms of newspapers, how much money would you receive?


In [6]:
# ── CELL 6: Load Qwen2.5-3B-Instruct ────────────────────────
# No 4-bit needed — 3B model in float16 uses ~6GB, safe on P100

print(f"Loading {CONFIG['model_name']}...")
print("Expected GPU memory: ~6GB in float16")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code = True,
)
tokenizer.padding_side = "left"    # critical for batch generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    dtype   = torch.float16,
    device_map    = "auto",
)
model.eval()

used_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nModel loaded!")
print(f"GPU used  : {used_gb:.2f} GB / {total_gb:.1f} GB")
print(f"Headroom  : {total_gb - used_gb:.1f} GB remaining")
print(f"Padding   : {tokenizer.padding_side} ✅")

if used_gb > 10:
    print("⚠️ WARNING: High memory usage. Reduce batch_size to 4 in Config if OOM errors occur.")
else:
    print("✅ Memory looks good.")

Loading Qwen/Qwen2.5-3B-Instruct...
Expected GPU memory: ~6GB in float16


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded!
GPU used  : 6.17 GB / 17.1 GB
Headroom  : 10.9 GB remaining
Padding   : left ✅
✅ Memory looks good.


In [7]:
# ── CELL 7: Prompt ────────────────────────────────────────────
# Qwen2.5 follows format constraints reliably
# Kept strict but cleaner than Phi version

SYSTEM_PROMPT = """You are a math problem decomposition assistant.
Your job: break a math problem into 2-5 numbered solution steps.

RULES:
1. Use format: Step 1: ... Step 2: ... etc.
2. Each step must be under 15 words.
3. Do NOT perform calculations or write numbers from computation.
4. Do NOT write the final answer.
5. Write ONLY the steps. Stop immediately after the last step.

EXAMPLE:
Problem: John earns $10/hour and works 8 hours. What does he earn?
Step 1: Identify the hourly rate and total hours worked.
Step 2: Multiply the hourly rate by the number of hours.
Step 3: The result is the total earnings."""

USER_TEMPLATE = """Problem: {question}"""


def build_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_TEMPLATE.format(question=question)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = True,
    )


print("Prompt preview:")
print(build_prompt("John has 5 apples and buys 3 more. How many total?"))

Prompt preview:
<|im_start|>system
You are a math problem decomposition assistant.
Your job: break a math problem into 2-5 numbered solution steps.

RULES:
1. Use format: Step 1: ... Step 2: ... etc.
2. Each step must be under 15 words.
3. Do NOT perform calculations or write numbers from computation.
4. Do NOT write the final answer.
5. Write ONLY the steps. Stop immediately after the last step.

EXAMPLE:
Problem: John earns $10/hour and works 8 hours. What does he earn?
Step 1: Identify the hourly rate and total hours worked.
Step 2: Multiply the hourly rate by the number of hours.
Step 3: The result is the total earnings.<|im_end|>
<|im_start|>user
Problem: John has 5 apples and buys 3 more. How many total?<|im_end|>
<|im_start|>assistant



In [8]:
# ── CELL 8: Cleaning + Filtering ─────────────────────────────

def clean_sg_output(text: str) -> str:
    """Extract only Step lines. Stop at first non-step line."""
    lines        = text.strip().split("\n")
    clean_lines  = []
    step_started = False

    for line in lines:
        line = line.strip()
        if not line:
            continue
        is_step = bool(re.match(
            r"^(Step\s*\d+[:.)]|\d+[.)]\s)", line, re.IGNORECASE
        ))
        if is_step:
            step_started = True
            # trim trailing garbage after first sentence
            trimmed = re.split(r"\.\s+[A-Z]", line)[0]
            if not trimmed.endswith("."):
                trimmed += "."
            clean_lines.append(trimmed)
        elif step_started:
            break

    return "\n".join(clean_lines)


def extract_steps(text: str) -> list:
    return [
        l.strip() for l in text.strip().split("\n")
        if re.match(r"^(Step\s*\d+[:.)]|\d+[.)]\s)", l.strip(), re.IGNORECASE)
    ]


def step_body(step: str) -> str:
    """Remove 'Step N:' prefix to get content only."""
    return re.sub(
        r"^(Step\s*\d+[:.)]|\d+[.)]\s)", "", step, flags=re.IGNORECASE
    ).strip()


def contains_calculation(text: str) -> bool:
    patterns = [
        r"\d+\s*[\+\-\×\÷\*\/]\s*\d+",
        r"=\s*\$?\d+",
        r"the answer is\s+\d+",
        r"\$\s*\d+\.?\d*",
        r"\\frac", r"\\times", r"\\left",
        r"\\\(", r"\\\)",
    ]
    return any(re.search(p, text, re.IGNORECASE) for p in patterns)


def steps_sequential(steps: list) -> bool:
    for i, step in enumerate(steps):
        m = re.match(r"^(?:Step\s*)(\d+)", step, re.IGNORECASE)
        if m and int(m.group(1)) != i + 1:
            return False
    return True


def steps_concise(steps: list, max_words: int) -> bool:
    return all(len(step_body(s).split()) <= max_words for s in steps)


def is_valid_sg(text: str) -> tuple:
    if not text or len(text.strip()) < 15:
        return False, "too_short"

    steps = extract_steps(text)

    if len(steps) < 2:                          return False, "too_few_steps"
    if len(steps) > 5:                          return False, "too_many_steps"
    if contains_calculation(text):              return False, "contains_calculation"
    if not steps_sequential(steps):             return False, "non_sequential_steps"
    if not steps_concise(steps, CONFIG["max_words_per_step"]):
                                                return False, "step_too_long"

    action_verbs = [
        "calculate", "determine", "identify", "find", "compute",
        "add", "subtract", "multiply", "divide", "sum", "count",
        "check", "compare", "convert", "evaluate", "use", "total",
        "measure", "estimate", "figure", "apply"
    ]
    if not any(v in text.lower() for v in action_verbs):
        return False, "no_action_verb"

    return True, "ok"


print("Filters loaded. Running sanity check...")
good = """Step 1: Identify the number of can groups in total collected.
Step 2: Calculate total money from recycling cans.
Step 3: Calculate total money from recycling newspapers.
Step 4: Add both amounts to get the total received."""

v, r = is_valid_sg(good)
steps = extract_steps(good)
print(f"Test SG → valid={v}, reason={r}, steps={len(steps)}")
for s in steps:
    print(f"  [{len(step_body(s).split()):2d}w] {s}")

Filters loaded. Running sanity check...
Test SG → valid=True, reason=ok, steps=4
  [ 9w] Step 1: Identify the number of can groups in total collected.
  [ 6w] Step 2: Calculate total money from recycling cans.
  [ 6w] Step 3: Calculate total money from recycling newspapers.
  [ 8w] Step 4: Add both amounts to get the total received.


In [9]:
# ── CELL 9: Generation + Single Test ─────────────────────────

def generate_sg_batch(questions: list) -> list:
    prompts = [build_prompt(q) for q in questions]

    inputs = tokenizer(
        prompts,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = CONFIG["max_new_tokens"],
            temperature        = CONFIG["temperature"],
            do_sample          = CONFIG["do_sample"],
            pad_token_id       = tokenizer.eos_token_id,
            eos_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.2,
        )

    input_len = inputs["input_ids"].shape[1]
    results   = []
    for output in outputs:
        raw   = tokenizer.decode(output[input_len:], skip_special_tokens=True).strip()
        clean = clean_sg_output(raw)
        results.append((raw, clean))
    return results


# ── Single test ──
print("=" * 55)
print("SINGLE SAMPLE TEST")
print("=" * 55)

test_q   = sampled_data[0]["question"]
test_out = generate_sg_batch([test_q])
raw, clean = test_out[0]

print(f"Question:\n{test_q}")
print(f"\nRaw output:\n{raw}")
print(f"\nCleaned SG:\n{clean}")

steps = extract_steps(clean)
valid, reason = is_valid_sg(clean)
print(f"\nResult → valid={valid} | reason={reason} | steps={len(steps)}")
for s in steps:
    words = len(step_body(s).split())
    flag  = "✅" if words <= CONFIG["max_words_per_step"] else "❌ TOO LONG"
    print(f"  [{words:2d}w] {flag}  {s}")

print("\n" + "=" * 55)
print("If steps look clean and valid=True → run Cell 10")
print("If still generating garbage → contact for further debugging")
print("=" * 55)

SINGLE SAMPLE TEST
Question:
For every 12 cans you recycle, you receive $0.50, and for every 5 kilograms of newspapers, you receive $1.50. If your family collected 144 cans and 20 kilograms of newspapers, how much money would you receive?

Raw output:
Step 1: Calculate the amount earned from recycling cans.
Step 2: Calculate the amount earned from recycling newspapers.
Step 3: Add both amounts to find total earnings.
Step 4: Verify calculation if necessary.

Cleaned SG:
Step 1: Calculate the amount earned from recycling cans.
Step 2: Calculate the amount earned from recycling newspapers.
Step 3: Add both amounts to find total earnings.
Step 4: Verify calculation if necessary.

Result → valid=True | reason=ok | steps=4
  [ 7w] ✅  Step 1: Calculate the amount earned from recycling cans.
  [ 7w] ✅  Step 2: Calculate the amount earned from recycling newspapers.
  [ 7w] ✅  Step 3: Add both amounts to find total earnings.
  [ 4w] ✅  Step 4: Verify calculation if necessary.

If steps look cle

In [10]:
# ── CELL 10: Main Loop ────────────────────────────────────────

print(f"Generating SG for {len(sampled_data)} questions...")
print("-" * 50)

results   = []
start_idx = 0

# Resume from checkpoint if session died
if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["raw_output_file"]):
        with open(CONFIG["raw_output_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed: idx={start_idx}, saved={len(results)}")
else:
    print("Starting fresh...")

for batch_start in tqdm(
    range(start_idx, len(sampled_data), CONFIG["batch_size"]),
    desc="Generating"
):
    batch_end = min(batch_start + CONFIG["batch_size"], len(sampled_data))
    batch     = sampled_data[batch_start:batch_end]
    questions = [b["question"] for b in batch]
    answers   = [b["answer"]   for b in batch]

    try:
        batch_out = generate_sg_batch(questions)
    except RuntimeError as e:
        print(f"OOM at batch {batch_start} — try reducing batch_size in Config")
        print(f"Error: {e}")
        break

    for i, (question, answer, (sg_raw, sg_clean)) in enumerate(
        zip(questions, answers, batch_out)
    ):
        gt_match  = re.search(r"####\s*(-?[\d,]+)", answer)
        gt_answer = gt_match.group(1).replace(",", "") if gt_match else ""
        valid, reason = is_valid_sg(sg_clean)
        steps         = extract_steps(sg_clean)

        results.append({
            "id"        : batch_start + i,
            "question"  : question,
            "sg_raw"    : sg_raw,
            "sg_clean"  : sg_clean,
            "sg_steps"  : steps,
            "gt_answer" : gt_answer,
            "valid"     : valid,
            "reason"    : reason,
        })

    # Checkpoint
    if len(results) % CONFIG["save_every"] < CONFIG["batch_size"]:
        with open(CONFIG["raw_output_file"], "w") as f:
            for item in results: f.write(json.dumps(item) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": batch_end}, f)

# Final save
with open(CONFIG["raw_output_file"], "w") as f:
    for item in results: f.write(json.dumps(item) + "\n")

valid_count = sum(1 for r in results if r["valid"])
print(f"\nTotal     : {len(results)}")
print(f"Valid     : {valid_count}")
print(f"Invalid   : {len(results) - valid_count}")
print(f"Valid rate: {valid_count/max(1,len(results))*100:.1f}%")

Generating SG for 3000 questions...
--------------------------------------------------
Starting fresh...


Generating:   0%|          | 0/375 [00:00<?, ?it/s]


Total     : 3000
Valid     : 2816
Invalid   : 184
Valid rate: 93.9%


In [11]:
# ── CELL 11: Save + Show Samples ─────────────────────────────

valid_results = [r for r in results if r["valid"]]

with open(CONFIG["valid_output_file"], "w") as f:
    for item in valid_results:
        f.write(json.dumps(item) + "\n")

print(f"Saved {len(valid_results)} valid SG → {CONFIG['valid_output_file']}")
print("\n" + "=" * 60)
print("SAMPLE VALID SG OUTPUTS")
print("=" * 60)

for i, item in enumerate(valid_results[:5]):
    print(f"\n--- Example {i+1} ---")
    print(f"Q : {item['question'][:100]}")
    print("SG:")
    for s in item["sg_steps"]:
        words = len(step_body(s).split())
        print(f"  [{words:2d}w] {s}")
    print(f"GT: {item['gt_answer']}")

Saved 2816 valid SG → /kaggle/working/sg_data/sg_valid.jsonl

SAMPLE VALID SG OUTPUTS

--- Example 1 ---
Q : For every 12 cans you recycle, you receive $0.50, and for every 5 kilograms of newspapers, you recei
SG:
  [ 6w] Step 1: Calculate money earned from recycling cans.
  [ 6w] Step 2: Calculate money earned from newspaper collection.
  [ 7w] Step 3: Add both amounts to find total received.
GT: 12

--- Example 2 ---
Q : Betty picked 16 strawberries. Matthew picked 20 more strawberries than Betty and twice as many as Na
SG:
  [ 6w] Step 1: Determine how many strawberries Matthew picked.
  [10w] Step 2: Calculate how many strawberries Natalie picked based on Matthew's amount.
  [10w] Step 3: Find the total number of strawberries all three collected together.
  [13w] Step 4: Compute how many jars of jam can be made with the total berries.
  [10w] Step 5: Calculate the total revenue from selling the jars of jam.
GT: 40

--- Example 3 ---
Q : Jack has a stack of books that is 12 inches t

In [12]:
# ── CELL 12: Report ───────────────────────────────────────────

print("=" * 60)
print("FINAL REPORT")
print("=" * 60)

valid_results = [r for r in results if r["valid"]]
failed        = [r for r in results if not r["valid"]]

# Step count distribution
step_counts = {}
for item in valid_results:
    n = len(item["sg_steps"])
    step_counts[n] = step_counts.get(n, 0) + 1

print(f"\nTotal generated : {len(results)}")
print(f"Valid           : {len(valid_results)} ({len(valid_results)/max(1,len(results))*100:.1f}%)")
print(f"Invalid         : {len(failed)}")

print("\nStep count distribution:")
for n, c in sorted(step_counts.items()):
    bar = "█" * max(1, c * 20 // max(1, len(valid_results)))
    print(f"  {n} steps: {c:3d} ({c/max(1,len(valid_results))*100:.0f}%) {bar}")

# Word count stats
all_words = [
    len(step_body(s).split())
    for item in valid_results
    for s in item["sg_steps"]
]
if all_words:
    print(f"\nStep word count:")
    print(f"  Average : {sum(all_words)/len(all_words):.1f} words (target <{CONFIG['max_words_per_step']})")
    print(f"  Min/Max : {min(all_words)} / {max(all_words)} words")

# Failure reasons
reasons = {}
for r in failed:
    reasons[r["reason"]] = reasons.get(r["reason"], 0) + 1
if reasons:
    print(f"\nFailure breakdown:")
    for reason, count in sorted(reasons.items(), key=lambda x: -x[1]):
        print(f"  {reason:25s}: {count}")

# Save
report = {
    "model"          : CONFIG["model_name"],
    "total"          : len(results),
    "valid"          : len(valid_results),
    "valid_rate"     : round(len(valid_results)/max(1,len(results)), 4),
    "step_dist"      : step_counts,
    "avg_step_words" : round(sum(all_words)/max(1,len(all_words)), 1) if all_words else 0,
    "failure_reasons": reasons,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(report, f, indent=2)

print(f"\nReport → {CONFIG['report_file']}")
print("\n" + "=" * 60)
print("NEXT STEPS:")
print("=" * 60)
vr = len(valid_results)/max(1,len(results))*100
if vr >= 70:
    print(f"✅ Valid rate {vr:.0f}% — GOOD!")
    print("   1. Visually check 5 sample SGs above — do steps look clean?")
    print("   2. If yes → set num_samples=3000 in Cell 4 and rerun Cell 10")
    print("   3. Full run takes ~1.5-2 hours on P100")
elif vr >= 40:
    print(f"⚠️  Valid rate {vr:.0f}% — acceptable but not ideal")
    print("   Share the failure breakdown and sample outputs for further tuning")
else:
    print(f"❌ Valid rate {vr:.0f}% — still a problem")
    print("   Share the Cell 9 single test output for debugging")
print("=" * 60)

FINAL REPORT

Total generated : 3000
Valid           : 2816 (93.9%)
Invalid         : 184

Step count distribution:
  2 steps:  47 (2%) █
  3 steps: 250 (9%) █
  4 steps: 305 (11%) ██
  5 steps: 2214 (79%) ███████████████

Step word count:
  Average : 8.3 words (target <20)
  Min/Max : 2 / 20 words

Failure breakdown:
  contains_calculation     : 173
  too_many_steps           : 7
  step_too_long            : 2
  no_action_verb           : 1
  too_few_steps            : 1

Report → /kaggle/working/sg_data/generation_report.json

NEXT STEPS:
✅ Valid rate 94% — GOOD!
   1. Visually check 5 sample SGs above — do steps look clean?
   2. If yes → set num_samples=3000 in Cell 4 and rerun Cell 10
   3. Full run takes ~1.5-2 hours on P100
